# PiShield — a Shield Layer for hierarchical requirements (C-HMCNN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihaela-stoian/PiShield/blob/hierarchical-requirements/examples/shield_layer_hierarchical.ipynb)

This notebook trains and tests a hierarchical multi-label classifier on the real
**cellcycle** functional-genomics dataset, reproducing **C-HMCNN** [3] inside PiShield.
It runs **end-to-end** on the dataset bundled with PiShield under `data/hierarchical_requirements/`.

The task is hierarchical multi-label classification: each gene belongs to functional
classes organised in a hierarchy, and **if a class is predicted, all of its ancestors must
be predicted too**. A hierarchical **Shield Layer** guarantees exactly that on every forward
pass, using C-HMCNN's *Max Constraint Module* (each class score becomes the maximum over its
own subtree). We train with it, evaluate, and compare against an unconstrained baseline.

> Requires `scikit-learn` (for standardisation and average precision). It is preinstalled on
> Colab; locally, `pip install scikit-learn`.

## Setup

Makes this notebook run both **locally** (from a repo checkout) and on **Google Colab**.
On Colab it clones the `hierarchical-requirements` branch — which brings both the PiShield
source and the bundled cellcycle datasets — and switches into it; locally it finds your
checkout automatically. `DATA_DIR` points at the cellcycle data and can be **overridden** to
read from somewhere else (e.g. your own upload, a mounted Google Drive, or the `cellcycle_GO`
folder to try the GO ontology, a DAG).

In [ ]:
import os, sys, subprocess

BRANCH = "hierarchical-requirements"
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

def _is_repo_root(d):
    return (os.path.isdir(os.path.join(d, "pishield")) and
            os.path.isdir(os.path.join(d, "data", "hierarchical_requirements")))

# Look for a local checkout by walking up from the working directory.
root = os.path.abspath(os.getcwd())
while root != os.path.dirname(root) and not _is_repo_root(root):
    root = os.path.dirname(root)

# No local checkout (e.g. a fresh Colab runtime): clone the branch (code + datasets).
if not _is_repo_root(root):
    if not os.path.isdir("PiShield"):
        subprocess.run(["git", "clone", "--branch", BRANCH,
                        "https://github.com/mihaela-stoian/PiShield.git"], check=True)
    root = os.path.abspath("PiShield")

os.chdir(root)
sys.path.insert(0, root)   # use this checkout's pishield/ (no install needed)

# Where the cellcycle data lives. Override to point at your own upload / Google Drive,
# or at the cellcycle_GO folder to try the GO ontology (a DAG) instead.
DATA_DIR = os.path.join(root, "data", "hierarchical_requirements", "cellcycle_FUN")
print(("Colab" if ON_COLAB else "local"), "| working from:", root)
print("DATA_DIR:", DATA_DIR)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

from pishield.shield_layer import build_shield_layer
from pishield.hierarchical_requirements.datasets import load_arff_dataset

torch.manual_seed(0)
np.random.seed(0)

# Quick demo by default (~2 min on CPU). Set FULL_REPRO = True for C-HMCNN's exact
# hyperparameters (hidden 500, lr 1e-4, Adam wd 1e-5, dropout 0.7, batch 4): it refits on
# train+val for `selected_epochs` and reproduces the paper's average precision, but is slow.
FULL_REPRO = False

## 1. Load the cellcycle dataset

`load_arff_dataset` reads an `.arff` file into features `X` and **ancestor-closed** labels
`Y` (if a class is labelled, so are all its ancestors), with the label columns ordered
exactly as the Shield Layer's classes. This is C-HMCNN's own data loader.

In [ ]:
dataset_name = os.path.basename(DATA_DIR)   # e.g. 'cellcycle_FUN'
train = load_arff_dataset(f"{DATA_DIR}/{dataset_name}.train.arff")
valid = load_arff_dataset(f"{DATA_DIR}/{dataset_name}.valid.arff")
test  = load_arff_dataset(f"{DATA_DIR}/{dataset_name}.test.arff")

num_classes = train.num_classes
to_eval = torch.tensor(train.to_eval)   # columns scored (root class excluded)
print(train)
print(test)
print("classes:", num_classes, "| scored:", int(to_eval.sum()))

## 2. Build the hierarchical Shield Layer

The layer reads the hierarchy from the same `.arff` header (auto-detecting the format) and
needs no manual class count — it adopts the parsed one. We check that its class order matches
the loaded labels, so `Y` lines up column-for-column with the output.

In [ ]:
shield = build_shield_layer(num_classes, f"{DATA_DIR}/{dataset_name}.train.arff")
assert shield.class_names == train.class_names

## 3. Preprocess the features

Standardise features (fit on train+val, as in C-HMCNN). The loader has already imputed the
missing values (`?`) with column means. We keep the train / validation split (used only to
*choose* the number of epochs in the optional cell below) and also form the combined
**train+val** set that the final model is trained on.

In [ ]:
scaler = StandardScaler().fit(np.vstack([train.X, valid.X]))

def X_tensor(a): return torch.tensor(scaler.transform(a), dtype=torch.float32)
def Y_tensor(a): return torch.tensor(a, dtype=torch.float32)

X_tr, Y_tr = X_tensor(train.X), Y_tensor(train.Y)
X_va, Y_va = X_tensor(valid.X), Y_tensor(valid.Y)
X_te, Y_te = X_tensor(test.X),  Y_tensor(test.Y)
X_trval = torch.cat([X_tr, X_va]); Y_trval = torch.cat([Y_tr, Y_va])
print("train:", tuple(X_tr.shape), "| train+val:", tuple(X_trval.shape), "| test:", tuple(X_te.shape))

## 4. Define the model

C-HMCNN's classifier: a 3-layer MLP with ReLU, dropout, and a sigmoid output (one score
per class). The Shield Layer is applied to that output — the model itself is unchanged.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.7):
        super().__init__()
        self.fc = nn.ModuleList([nn.Linear(in_dim, hidden),
                                 nn.Linear(hidden, hidden),
                                 nn.Linear(hidden, out_dim)])
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for layer in self.fc[:-1]:
            x = self.drop(self.relu(layer(x)))
        return self.sigmoid(self.fc[-1](x))

in_dim = X_tr.shape[1]
hidden = 500   # C-HMCNN cellcycle_FUN (it uses a per-dataset hidden size; see its main.py)

def train_model(X, Y, use_shield, epochs, batch_size, lr):
    model = MLP(in_dim, hidden, num_classes)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5, betas=(0.9, 0.999))
    bce = nn.BCELoss()
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, Y),
                                         batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            out = model(xb)
            if use_shield:
                out = shield(out, goal=yb)   # goal-masked MCM (C-HMCNN's max-constraint loss)
            loss = bce(out[:, to_eval], yb[:, to_eval])
            loss.backward()
            opt.step()
    return model

## (Optional) Choosing the number of epochs

The paper picks the number of training epochs by training on the **training set** and
early-stopping on the **validation set** (patience 20); the final model is then retrained on
train+val for that many epochs. For **cellcycle_FUN** that number is already known — **106** —
so you can leave this cell as-is and skip straight to training.

If you point `DATA_DIR` at a **different FUN/GO dataset**, set `RUN_EPOCH_SELECTION = True` to
search for *its* best epoch (also set `hidden` above to that dataset's size — see C-HMCNN's
`main.py`). This runs the selection at batch size 4, so it is **slow**.

In [ ]:
RUN_EPOCH_SELECTION = False    # True to search for the best epoch on the current dataset
selected_epochs = 106          # known best for cellcycle_FUN; overwritten if selection runs

if RUN_EPOCH_SELECTION:
    MAX_EPOCHS, PATIENCE = 200, 20
    def val_ap(model):
        model.eval()
        with torch.no_grad():
            p = shield(model(X_va))
        return average_precision_score(Y_va[:, to_eval].numpy(),
                                       p[:, to_eval].numpy(), average="micro")
    m = MLP(in_dim, hidden, num_classes)
    opt = torch.optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-5, betas=(0.9, 0.999))
    bce = nn.BCELoss()
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_tr, Y_tr),
                                         batch_size=4, shuffle=True)
    best_ap, selected_epochs, waited = -1.0, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        m.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = bce(shield(m(xb), goal=yb)[:, to_eval], yb[:, to_eval])
            loss.backward(); opt.step()
        ap = val_ap(m)
        if ap > best_ap + 1e-5:
            best_ap, selected_epochs, waited = ap, epoch, 0
        else:
            waited += 1
        print(f"epoch {epoch}: val AP={ap:.4f}  (best {best_ap:.4f} @ epoch {selected_epochs})")
        if waited >= PATIENCE:
            break

print("selected_epochs =", selected_epochs)

## 5. Train the final model (on train+val)

We train on **train+val** for `selected_epochs`, following C-HMCNN. At training time we pass
the labels as `goal`, so the layer applies the Max Constraint Module in its *max-constraint
loss* form (`(1 - goal) * MCM(pred) + goal * MCM(goal * pred)`), pushing the model to raise the
relevant descendant rather than the ancestor. Gradients flow through the layer.

In [ ]:
if FULL_REPRO:
    epochs, batch_size, lr = selected_epochs, 4, 1e-4   # C-HMCNN cellcycle_FUN
else:
    epochs, batch_size, lr = 10, 128, 2e-3              # quick demo
print(f"Training ({'FULL_REPRO' if FULL_REPRO else 'quick'}): "
      f"epochs={epochs}, batch={batch_size}, lr={lr}")
model = train_model(X_trval, Y_trval, use_shield=True, epochs=epochs, batch_size=batch_size, lr=lr)

## 6. Evaluate

At inference the layer applies the plain Max Constraint Module (no `goal`). We report the
micro-averaged **average precision** on the scored classes — C-HMCNN's metric — and count
**hierarchy violations** at the score level: edges where the predicted `score(child)` exceeds
`score(parent)`. The Max Constraint Module guarantees this is **exactly zero** for the
shielded model, for any input.

In [ ]:
edges = shield.hierarchy.edges
n_edge_instances = len(edges) * len(X_te)

def hierarchy_violations(preds):
    bad = 0
    for child, parent in edges:
        bad += int((preds[:, child] > preds[:, parent] + 1e-6).sum())
    return bad

def evaluate(model, apply_shield):
    model.eval()
    with torch.no_grad():
        preds = model(X_te)
        if apply_shield:
            preds = shield(preds)
    ap = average_precision_score(Y_te[:, to_eval].numpy(),
                                 preds[:, to_eval].numpy(), average="micro")
    return ap, hierarchy_violations(preds)

ap, viol = evaluate(model, apply_shield=True)
print(f"shielded : test AP(micro)={ap:.4f}, violated edges={viol}/{n_edge_instances}")

## 7. Compare against an unconstrained baseline

The same MLP trained (on train+val) without the Shield Layer — plain BCE, no correction at
inference.

In [ ]:
base = train_model(X_trval, Y_trval, use_shield=False, epochs=epochs, batch_size=batch_size, lr=lr)
base_ap, base_viol = evaluate(base, apply_shield=False)
print(f"baseline : test AP(micro)={base_ap:.4f}, violated edges={base_viol}/{n_edge_instances}")
print(f"shielded : test AP(micro)={ap:.4f}, violated edges={viol}/{n_edge_instances}")

## 8. Takeaway

The **shielded** model is hierarchically coherent **by construction** — zero violated edges,
for any input — while the unconstrained baseline scores many descendants above their
ancestors. With `FULL_REPRO = True` it follows C-HMCNN's protocol (refit on train+val for the
selected epochs) and matches the average precision reported for cellcycle_FUN in [3]; the
quick default trades some accuracy for speed but shows the same guarantee.

To reuse this on another **FUN** or **GO** dataset, point `DATA_DIR` at its folder, set
`hidden` to that dataset's size, and run the optional epoch-selection cell to find its best
epoch. GO ontologies are *DAGs* (a class can have several parents) and have ~4000 classes, so
use the `FULL_REPRO` batch size of 4 to keep the Max Constraint Module's memory modest.